# EquiTwin Pune/PMPML — Data Analysis Notebook
**Member 4 | Analytics, Visualization & IEEE Documentation Lead**  
Dataset Year: 2025 (simulation baseline) | Random Seed: 42

This notebook performs statistical analysis on all EquiTwin datasets:
- `traffic_conditions.csv` — per-route, per-time-slot congestion/speed/delay
- `incidents.csv` — weekday disruption events
- `zones.csv` — socio-economic zone mapping
- GTFS files: `routes.txt`, `stops.txt`, `trips.txt`, `stop_times.txt`

---


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Path configuration ──────────────────────────────────────
DATA_DIR   = "."          # change if running from a different working directory
GTFS_DIR   = "."
TRAFFIC_F  = f"{DATA_DIR}/traffic_conditions.csv"
INCIDENTS_F= f"{DATA_DIR}/incidents.csv"
ZONES_F    = f"{DATA_DIR}/zones.csv"
ROUTES_F   = f"{GTFS_DIR}/routes.txt"
STOPS_F    = f"{GTFS_DIR}/stops.txt"
TRIPS_F    = f"{GTFS_DIR}/trips.txt"
STOP_TIMES_F = f"{GTFS_DIR}/stop_times.txt"
CALENDAR_F = f"{GTFS_DIR}/calendar.txt"

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

np.random.seed(42)
print("Libraries loaded. Ready.")


In [ ]:
# ── Load all datasets ───────────────────────────────────────
traffic   = pd.read_csv(TRAFFIC_F)
incidents = pd.read_csv(INCIDENTS_F)
zones     = pd.read_csv(ZONES_F)
routes    = pd.read_csv(ROUTES_F)
stops     = pd.read_csv(STOPS_F)
trips     = pd.read_csv(TRIPS_F)
stop_times= pd.read_csv(STOP_TIMES_F)
calendar  = pd.read_csv(CALENDAR_F)

print(f"traffic_conditions : {traffic.shape}")
print(f"incidents          : {incidents.shape}")
print(f"zones              : {zones.shape}")
print(f"routes             : {routes.shape}")
print(f"stops              : {stops.shape}")
print(f"trips              : {trips.shape}")
print(f"stop_times         : {stop_times.shape}")
print(f"calendar           : {calendar.shape}")


## 2. Dataset Inspection

In [ ]:
print("=== traffic_conditions.csv ===")
display(traffic.head(10))
print("\nData types:")
print(traffic.dtypes)


In [ ]:
print("=== incidents.csv ===")
display(incidents.head(10))


In [ ]:
print("=== zones.csv ===")
display(zones.head())


In [ ]:
print("=== routes.txt ===")
display(routes.head())
print("\n=== stops.txt ===")
display(stops.head())


## 3. Null & Integrity Checks

In [ ]:
datasets = {
    'traffic_conditions': traffic,
    'incidents'         : incidents,
    'zones'             : zones,
    'routes'            : routes,
    'stops'             : stops,
    'trips'             : trips,
    'stop_times'        : stop_times,
}

print(f"{'Dataset':<25} {'Rows':>7} {'Cols':>5} {'Nulls':>7} {'Null%':>7}")
print("-" * 55)
for name, df in datasets.items():
    nulls = df.isnull().sum().sum()
    pct   = 100 * nulls / (df.shape[0] * df.shape[1])
    print(f"{name:<25} {df.shape[0]:>7} {df.shape[1]:>5} {nulls:>7} {pct:>6.1f}%")


In [ ]:
# Duplicate ID checks
print("Duplicate route_ids in routes.txt :", routes['route_id'].duplicated().sum())
print("Duplicate stop_ids in stops.txt   :", stops['stop_id'].duplicated().sum())
print("Duplicate trip_ids in trips.txt   :", trips['trip_id'].duplicated().sum())

# Check traffic coverage: should have 10 routes × 7 slots = 70 rows
expected_rows = traffic['route_id'].nunique() * traffic['time_slot'].nunique()
print(f"\ntraffic_conditions expected rows   : {expected_rows}")
print(f"traffic_conditions actual rows     : {len(traffic)}")
print(f"Coverage complete                  : {len(traffic) == expected_rows}")


In [ ]:
# Validate speed & delay bounds per congestion_assumptions.md
speed_ok  = traffic['average_speed_kmh'].between(8.0, 48.0).all()
delay_ok  = traffic['delay_multiplier'].between(1.0, 3.0).all()
cong_ok   = traffic['congestion_level'].between(5, 145).all()

print(f"Speed in [8.0, 48.0] km/h : {speed_ok}")
print(f"Delay in [1.0, 3.0]       : {delay_ok}")
print(f"Congestion in [5, 145]%   : {cong_ok}")

print(f"\nSpeed min/max: {traffic['average_speed_kmh'].min():.1f} / {traffic['average_speed_kmh'].max():.1f} km/h")
print(f"Delay min/max: {traffic['delay_multiplier'].min():.2f} / {traffic['delay_multiplier'].max():.2f}")
print(f"Congestion min/max: {traffic['congestion_level'].min():.1f} / {traffic['congestion_level'].max():.1f}%")


## 4. Traffic Conditions — Statistical Analysis

In [ ]:
print("=== Descriptive Statistics: traffic_conditions ===")
display(traffic[['congestion_level','average_speed_kmh','delay_multiplier']].describe().T)


In [ ]:
# ── Speed by time slot ──────────────────────────────────────
slot_order = ['night_00_06','early_am_06_07','peak_am_07_10',
              'midday_10_16','peak_pm_17_20','evening_20_23','late_night_23_24']

slot_stats = traffic.groupby('time_slot').agg(
    mean_speed   =('average_speed_kmh','mean'),
    std_speed    =('average_speed_kmh','std'),
    mean_cong    =('congestion_level','mean'),
    mean_delay   =('delay_multiplier','mean'),
    count        =('route_id','count')
).reindex(slot_order).round(2)

print("=== Speed / Congestion / Delay by Time Slot ===")
display(slot_stats)


In [ ]:
# ── BRTS vs Non-BRTS comparison ─────────────────────────────
brts_cmp = traffic.groupby('route_type').agg(
    mean_speed   =('average_speed_kmh','mean'),
    median_speed =('average_speed_kmh','median'),
    mean_cong    =('congestion_level','mean'),
    mean_delay   =('delay_multiplier','mean'),
    n            =('route_id','count')
).round(3)

print("=== BRTS vs Non-BRTS (all time slots) ===")
display(brts_cmp)

brts_speed    = traffic[traffic['route_type']=='BRTS']['average_speed_kmh']
nonbrts_speed = traffic[traffic['route_type']=='Non-BRTS']['average_speed_kmh']
t_stat, p_val = stats.ttest_ind(brts_speed, nonbrts_speed)
print(f"\nIndependent t-test — BRTS vs Non-BRTS speed:")
print(f"  t-statistic = {t_stat:.3f}  |  p-value = {p_val:.4f}")
print(f"  Result: {'Statistically significant (p<0.05)' if p_val<0.05 else 'Not significant'}")


In [ ]:
# ── Peak-hour BRTS advantage ────────────────────────────────
peak_slots = ['peak_am_07_10','peak_pm_17_20']
peak_df = traffic[traffic['time_slot'].isin(peak_slots)]

peak_adv = peak_df.groupby(['time_slot','route_type'])['average_speed_kmh'].mean().unstack()
peak_adv['BRTS_advantage_kmh'] = (peak_adv['BRTS'] - peak_adv['Non-BRTS']).round(2)
print("=== BRTS Speed Advantage During Peak Hours ===")
display(peak_adv.round(2))


In [ ]:
# ── Congestion label distribution ───────────────────────────
cong_dist = traffic['congestion_level_label'].value_counts(normalize=True).mul(100).round(1)
print("=== Congestion Label Distribution (%) ===")
print(cong_dist.to_string())


In [ ]:
# ── Route-level summary ─────────────────────────────────────
route_summary = traffic.groupby('route_id').agg(
    mean_speed   =('average_speed_kmh','mean'),
    min_speed    =('average_speed_kmh','min'),
    max_speed    =('average_speed_kmh','max'),
    mean_cong    =('congestion_level','mean'),
    mean_delay   =('delay_multiplier','mean'),
    type         =('route_type','first')
).round(2).sort_values('mean_speed')

print("=== Route-Level Performance Summary (sorted by mean speed) ===")
display(route_summary)


In [ ]:
# ── Correlation analysis ─────────────────────────────────────
num_cols = ['congestion_level','average_speed_kmh','delay_multiplier']
corr_matrix = traffic[num_cols].corr().round(3)
print("=== Pearson Correlation Matrix ===")
display(corr_matrix)

r_sc, p_sc = stats.pearsonr(traffic['congestion_level'], traffic['average_speed_kmh'])
r_sd, p_sd = stats.pearsonr(traffic['average_speed_kmh'], traffic['delay_multiplier'])
print(f"\nCongestion ↔ Speed   : r = {r_sc:.3f}  (p = {p_sc:.2e})")
print(f"Speed      ↔ Delay   : r = {r_sd:.3f}  (p = {p_sd:.2e})")


## 5. Incident Analysis

In [ ]:
print("=== Descriptive Stats: incidents ===")
display(incidents[['duration_minutes']].describe().T)

print("\n=== Incident Type Distribution ===")
type_dist = incidents['incident_type'].value_counts()
print(type_dist.to_frame('count').assign(pct=lambda x: (x['count']/len(incidents)*100).round(1)))


In [ ]:
# ── Severity split ───────────────────────────────────────────
sev_overall = incidents['severity'].value_counts()
print("=== Overall Severity Split ===")
print(sev_overall.to_frame('count').assign(pct=lambda x: (x['count']/len(incidents)*100).round(1)))

print("\n=== Severity Split by Incident Type ===")
sev_by_type = incidents.groupby(['incident_type','severity']).size().unstack(fill_value=0)
display(sev_by_type)


In [ ]:
# ── Duration analysis ────────────────────────────────────────
dur_by_type = incidents.groupby('incident_type')['duration_minutes'].agg(
    ['mean','median','min','max','std']
).round(1).sort_values('mean', ascending=False)
print("=== Duration (minutes) by Incident Type ===")
display(dur_by_type)

print(f"\nOverall mean duration : {incidents['duration_minutes'].mean():.1f} min")
print(f"Overall median        : {incidents['duration_minutes'].median():.1f} min")
print(f"Std deviation         : {incidents['duration_minutes'].std():.1f} min")


In [ ]:
# ── Route-level incident exposure ────────────────────────────
inc_by_route = incidents['route_id'].value_counts().reset_index()
inc_by_route.columns = ['route_id','incident_count']
print("=== Incident Count by Route ===")
print(inc_by_route.to_string(index=False))

routes_with_zero = set(traffic['route_id'].unique()) - set(incidents['route_id'].unique())
print(f"\nRoutes with 0 incidents: {routes_with_zero}")


In [ ]:
# ── Peak vs off-peak incident timing ────────────────────────
# Extract hour from start_time
incidents['start_hour'] = pd.to_datetime(incidents['start_time'], format='%H:%M').dt.hour

peak_hours  = incidents[incidents['start_hour'].isin(range(7,10))  |
                         incidents['start_hour'].isin(range(17,20))]
offpeak_hrs = incidents[~incidents.index.isin(peak_hours.index)]

print(f"Incidents during peak hours    : {len(peak_hours)} ({100*len(peak_hours)/len(incidents):.1f}%)")
print(f"Incidents during off-peak      : {len(offpeak_hrs)} ({100*len(offpeak_hrs)/len(incidents):.1f}%)")

dur_t, dur_p = stats.ttest_ind(peak_hours['duration_minutes'], offpeak_hrs['duration_minutes'])
print(f"\nDuration — peak vs off-peak t-test:")
print(f"  Peak mean : {peak_hours['duration_minutes'].mean():.1f} min")
print(f"  Off-peak mean : {offpeak_hrs['duration_minutes'].mean():.1f} min")
print(f"  t = {dur_t:.3f}  |  p = {dur_p:.4f}")


## 6. Zone & Equity Analysis

In [ ]:
print("=== zones.csv descriptive stats ===")
num_zone_cols = [c for c in zones.columns if zones[c].dtype in ['float64','int64']]
display(zones[num_zone_cols].describe().T.round(2))


In [ ]:
# ── Income group comparisons ─────────────────────────────────
income_grp = zones.groupby('income_group').agg(
    zones_count             =('zone_id','count'),
    avg_population          =('population','mean'),
    avg_density             =('population_density','mean'),
    avg_transport_dependency=('transport_dependency','mean'),
    avg_reliability         =('reliability_score','mean') if 'reliability_score' in zones.columns else ('population','count')
).round(2)

print("=== Zone Metrics by Income Group ===")
display(income_grp)


In [ ]:
# ── One-way ANOVA: reliability across income groups ─────────
if 'reliability_score' in zones.columns:
    groups = [zones[zones['income_group']==g]['reliability_score'] for g in zones['income_group'].unique()]
    f_stat, p_anova = stats.f_oneway(*groups)
    print(f"ANOVA — reliability_score across income groups:")
    print(f"  F = {f_stat:.3f}  |  p = {p_anova:.4f}")
    print(f"  Result: {'Significant group differences (p<0.05)' if p_anova<0.05 else 'No significant difference'}")
else:
    print("reliability_score column not found — skipping ANOVA.")


In [ ]:
# ── Transport dependency vs population density correlation ───
dep_col  = 'transport_dependency'
dens_col = 'population_density'

if dep_col in zones.columns and dens_col in zones.columns:
    r, p = stats.pearsonr(zones[dep_col], zones[dens_col])
    print(f"Correlation — transport dependency vs population density:")
    print(f"  r = {r:.3f}  |  p = {p:.4f}")


## 7. GTFS Network Analysis

In [ ]:
# ── Route network summary ────────────────────────────────────
print(f"Total routes  : {routes['route_id'].nunique()}")
print(f"Total stops   : {stops['stop_id'].nunique()}")
print(f"Total trips   : {trips['trip_id'].nunique()}")
print(f"Total stop_times: {len(stop_times)}")

if 'route_type' in routes.columns:
    print("\nRoute type distribution:")
    print(routes['route_type'].value_counts().to_string())


In [ ]:
# ── Trips per route ──────────────────────────────────────────
trips_per_route = trips.groupby('route_id').size().sort_values(ascending=False)
print("=== Top 10 Routes by Trip Count ===")
print(trips_per_route.head(10).to_string())
print(f"\nMean trips per route  : {trips_per_route.mean():.1f}")
print(f"Median trips per route: {trips_per_route.median():.1f}")


In [ ]:
# ── Stop-times: stops per trip ───────────────────────────────
stops_per_trip = stop_times.groupby('trip_id')['stop_sequence'].count()
print(f"Mean stops per trip   : {stops_per_trip.mean():.1f}")
print(f"Median stops per trip : {stops_per_trip.median():.1f}")
print(f"Min stops             : {stops_per_trip.min()}")
print(f"Max stops             : {stops_per_trip.max()}")


In [ ]:
# ── Service coverage by day ──────────────────────────────────
day_cols = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']
service_days = calendar[day_cols].sum().sort_values(ascending=False)
print("=== Number of services running on each day ===")
print(service_days.to_string())


## 8. Equity Metrics — Waiting Time Disparity

In [ ]:
# ── Derive a synthetic equity metric ─────────────────────────
# Waiting time ~ f(reliability, transport_dependency)
# Travel Time = Base Time × Congestion Factor + Incident Delay (EquiTwin formula)

if 'reliability_score' in zones.columns and 'transport_dependency' in zones.columns:
    # Normalise delay: use mean delay from traffic × mean incident duration from incidents
    mean_delay = traffic['delay_multiplier'].mean()
    mean_inc_duration_hr = incidents['duration_minutes'].mean() / 60

    zones['estimated_wait_peak_min'] = (
        (10 / (zones['reliability_score'] / 100)) * mean_delay +
        mean_inc_duration_hr * 60 * zones['transport_dependency'] * 0.15
    ).round(1)

    equity_by_income = zones.groupby('income_group')['estimated_wait_peak_min'].agg(['mean','min','max','std']).round(2)
    print("=== Estimated Peak Waiting Time by Income Group (minutes) ===")
    display(equity_by_income)

    high_wait = zones[zones['income_group']=='High']['estimated_wait_peak_min'].mean()
    low_wait  = zones[zones['income_group']=='Low']['estimated_wait_peak_min'].mean()
    print(f"\nTransit Inequality Gap (Low − High income) : {low_wait - high_wait:.1f} minutes")
    print(f"Relative disadvantage of Low-income zones  : {(low_wait/high_wait - 1)*100:.1f}% longer wait")
else:
    print("Required columns not found. Check zones.csv schema.")


In [ ]:
# ── Gini-like inequality index ────────────────────────────────
if 'estimated_wait_peak_min' in zones.columns:
    waits = zones['estimated_wait_peak_min'].sort_values().values
    n = len(waits)
    gini = (2 * np.sum(np.arange(1, n+1) * waits) / (n * np.sum(waits))) - (n+1)/n
    print(f"Gini coefficient of waiting time distribution: {gini:.3f}")
    print(f"(0 = perfect equality, 1 = total inequality)")
    print(f"\nMost equitable zone: {zones.loc[zones['estimated_wait_peak_min'].idxmin(), 'ward_name']}")
    print(f"Least equitable zone: {zones.loc[zones['estimated_wait_peak_min'].idxmax(), 'ward_name']}")


## 9. Summary & Key Findings

In [ ]:
print("=" * 60)
print("     EQUITWIN ANALYSIS — KEY FINDINGS SUMMARY")
print("=" * 60)

print("\n📊 TRAFFIC & CONGESTION")
print(f"  • City avg speed (TomTom 2025) : 19 km/h")
peak_am_speed = traffic[traffic['time_slot']=='peak_am_07_10']['average_speed_kmh'].mean()
peak_pm_speed = traffic[traffic['time_slot']=='peak_pm_17_20']['average_speed_kmh'].mean()
print(f"  • Simulated peak AM avg speed  : {peak_am_speed:.1f} km/h")
print(f"  • Simulated peak PM avg speed  : {peak_pm_speed:.1f} km/h")
print(f"  • Max delay multiplier observed: {traffic['delay_multiplier'].max():.2f}×")

brts_mean = traffic[traffic['route_type']=='BRTS']['average_speed_kmh'].mean()
nb_mean   = traffic[traffic['route_type']=='Non-BRTS']['average_speed_kmh'].mean()
print(f"  • BRTS vs Non-BRTS speed gap   : {brts_mean - nb_mean:.2f} km/h")

print("\n⚠️  INCIDENTS")
print(f"  • Total incidents              : {len(incidents)}")
top_type = incidents['incident_type'].value_counts().idxmax()
top_pct  = incidents['incident_type'].value_counts(normalize=True).max()*100
print(f"  • Dominant type                : {top_type} ({top_pct:.0f}%)")
print(f"  • Mean duration                : {incidents['duration_minutes'].mean():.0f} min")

print("\n⚖️  EQUITY")
if 'estimated_wait_peak_min' in zones.columns:
    print(f"  • Waiting time gap (Low−High)  : {low_wait - high_wait:.1f} min")
    print(f"  • Gini coefficient             : {gini:.3f}")
    print(f"  • Zones above 0.8 dependency   : {(zones['transport_dependency']>0.8).sum()}")

print("\n🗺️  NETWORK")
print(f"  • Routes in GTFS               : {routes['route_id'].nunique()}")
print(f"  • Stops in GTFS                : {stops['stop_id'].nunique()}")
print(f"  • Trips in GTFS                : {trips['trip_id'].nunique()}")
print("=" * 60)


---
*Generated for EquiTwin (Pune/PMPML) — Member 4 | Data Analysis Notebook*  
*Python 3.10 | pandas | numpy | scipy | Random Seed 42*
